In [1]:
import scanpy as sc 
import numpy as np
import pandas as pd

In [2]:
# %pip install scrublet

In [3]:
import scrublet as scr 
import os

In [4]:
# 4 TOTAL STUDIES

In [5]:
study_files = {
    "Chen2021": {
        "matrix": "Data_Chen2021_Prostate/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Chen2021_Prostate/Genes.txt",
        "cells":  "Data_Chen2021_Prostate/Cells.csv",
        "samples": "Data_Chen2021_Prostate/Samples.csv",
        "units": "UMI"
    },
    "Dong2020": {
        "matrix": "Data_Dong2020_Prostate/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Dong2020_Prostate/Genes.txt",
        "cells":  "Data_Dong2020_Prostate/Cells.csv",
        "samples": "Data_Dong2020_Prostate/Samples.csv",
        "units": "UMI"
    },
    "He2021": {
        "matrix": "Data_He2021_Prostate/Exp_data_TPM.mtx",
        "genes":  "Data_He2021_Prostate/Genes.txt",
        "cells":  "Data_He2021_Prostate/Cells.csv",
        "samples": "Data_He2021_Prostate/Samples.csv",
        "units": "TPM"
    },
    "Song2022": {
        "matrix": "Data_Song2022_Prostate/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Song2022_Prostate/Genes.txt",
        "cells":  "Data_Song2022_Prostate/Cells.csv",
        "samples": "Data_Song2022_Prostate/Samples.csv",
        "units": "UMI"
    }
}

In [6]:
# STUDY 1

In [7]:
study_name = 'Chen2021'
paths = study_files[study_name]

In [8]:
pwd

'/home/pandeyps/Prefix/scM/quality control/prostate'

In [9]:
# creating matrix (cells x gene)
adata = sc.read_mtx(paths['matrix']).T

In [10]:
# gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes

In [11]:
# cell metadata
cells = pd.read_csv(paths["cells"], index_col=0)
adata.obs_names = cells.index
adata.obs_names = adata.obs_names.astype(str) 
adata.obs = adata.obs.join(cells, how="left")

In [12]:
# adding study column
adata.obs["study"] = study_name

In [13]:
f'{adata.n_obs:,} cells, {adata.n_vars:,} genes'

'36,424 cells, 25,044 genes'

In [14]:
# METADATA - DROP NaN Values

In [15]:
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease'] #check if present

# combine all cols that are in the study
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]

drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    
    # 1. NaN (for any dtype)
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask = drop_mask | nan_mask
    
    # 2. For object / string columns, "unknown", "NA", "nan", empty
    if vals.dtype == object:
        # Convert to string, strip and lowercase
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} values to be dropped")
            drop_mask = drop_mask | suspect

print(f"cells flagged: {drop_mask.sum()}")

# drop
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"cells remaining after drop: {adata.n_obs}")
else:
    print('no cells dropped')

   cell_type: 5997 NaN
   cell_type: 5997 values to be dropped
cells flagged: 5997
cells remaining after drop: 30427


In [16]:
f'{adata.n_obs:,} cells, {adata.n_vars:,} genes'

'30,427 cells, 25,044 genes'

In [17]:
# GENE COUNT FILTER (200-6000)
sc.pp.calculate_qc_metrics(adata, inplace=True)

min_genes = 200
max_genes = 6000
keep = (adata.obs["n_genes_by_counts"] >= min_genes) & \
       (adata.obs["n_genes_by_counts"] <= max_genes)
adata = adata[keep, :].copy()

In [18]:
f" {adata.n_obs:,} cells"

' 30,427 cells'

In [19]:
# DOUBLET REMOVAL USING SCRUBLET
# scrublet needs dense array, our matrix is sparse, so converting it to an array
counts = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X

In [20]:
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.48
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 16.0%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.9%
Elapsed time: 50.5 seconds


In [21]:
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets

In [22]:
adata = adata[~adata.obs["predicted_doublet"], :].copy()

In [23]:
f"   After doublet removal- {adata.n_obs:,} cells"

'   After doublet removal- 30,385 cells'

In [24]:
# Dead cell removal
adata.var['mt'] = adata.var_names.str.startswith('MT-') #get mitochondrial genes
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
max_pct_mt = 15
keep_mt = adata.obs['pct_counts_mt'] <= max_pct_mt
adata = adata[keep_mt, :].copy()

In [25]:
f"after MT filter: {adata.n_obs:,} cells"

'after MT filter: 30,385 cells'

In [26]:
# Convert columns to string (conflict for h5ad)
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

In [27]:
adata.write_h5ad('prostate_study1_qc.h5ad')

In [29]:
# STUDY 2

In [30]:
study_name = 'Dong2020'
paths = study_files[study_name]

# matrix,gene and cell
adata = sc.read_mtx(paths['matrix']).T
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes

cells = pd.read_csv(paths["cells"], index_col=0)
adata.obs_names = cells.index
adata.obs_names = adata.obs_names.astype(str) 
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"{adata.n_obs:,} cells, {adata.n_vars:,} genes")

21,292 cells, 15,709 genes


In [31]:
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease'] #check if present

# combine all cols that are in the study
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]

drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    
    # 1. NaN (for any dtype)
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask = drop_mask | nan_mask
    
    # 2. For object / string columns, "unknown", "NA", "nan", empty
    if vals.dtype == object:
        # Convert to string, strip and lowercase
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} values to be dropped")
            drop_mask = drop_mask | suspect

print(f"cells flagged: {drop_mask.sum()}")

# drop
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"cells remaining after drop: {adata.n_obs}")
else:
    print('no cells dropped')

   cell_type: 6316 NaN
   cell_type: 6316 values to be dropped
cells flagged: 6316
cells remaining after drop: 14976


In [32]:
# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
min_genes = 200
max_genes = 6000
keep = (adata.obs["n_genes_by_counts"] >= min_genes) & \
       (adata.obs["n_genes_by_counts"] <= max_genes)
adata = adata[keep, :].copy()
print(f"after gene filter {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"after doublet removal: {adata.n_obs:,} cells")

# Dead cell removal
adata.var['mt'] = adata.var_names.str.startswith('MT-') #get mitochondrial genes
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
max_pct_mt = 15
keep_mt = adata.obs['pct_counts_mt'] <= max_pct_mt
adata = adata[keep_mt, :].copy()
print(f"after MT filter: {adata.n_obs:,} cells")

# conversion to string
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

adata.write_h5ad('prostate_study2_qc.h5ad')

after gene filter 14,976 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.48
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 35.6%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.4%
Elapsed time: 21.3 seconds
after doublet removal: 14,957 cells
after MT filter: 14,888 cells


In [39]:
# STUDY 3

In [ ]:
study_name = "He2021"
paths = study_files[study_name]

adata = sc.read_mtx(paths["matrix"]).T
adata.var_names = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)              

adata.obs_names = cells.index                      
adata.obs = adata.obs.join(cells, how="left")      

adata.obs["study"] = study_name
print(f"{adata.n_obs:,} cells, {adata.n_vars:,} genes")

2,170 cells, 45,895 genes


In [44]:
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease'] #check if present

# combine all cols that are in the study
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]

drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    
    # 1. NaN (for any dtype)
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask = drop_mask | nan_mask
    
    # 2. For object / string columns, "unknown", "NA", "nan", empty
    if vals.dtype == object:
        # Convert to string, strip and lowercase
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} values to be dropped")
            drop_mask = drop_mask | suspect

print(f"cells flagged: {drop_mask.sum()}")

# drop
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"cells remaining after drop: {adata.n_obs}")
else:
    print('no cells dropped')

cells flagged: 0
no cells dropped


In [46]:
# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
min_genes = 200
max_genes = 6000
keep = (adata.obs["n_genes_by_counts"] >= min_genes) & \
       (adata.obs["n_genes_by_counts"] <= max_genes)
adata = adata[keep, :].copy()
print(f"after gene filter {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"after doublet removal: {adata.n_obs:,} cells")

# Dead cell removal
adata.var['mt'] = adata.var_names.str.startswith('MT-') #get mitochondrial genes
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
max_pct_mt = 15
keep_mt = adata.obs['pct_counts_mt'] <= max_pct_mt
adata = adata[keep_mt, :].copy()
print(f"after MT filter: {adata.n_obs:,} cells")

# conversion to string
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

adata.write_h5ad('prostate_study3_qc.h5ad')

after gene filter 2,151 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.42
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 43.0%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.0%
Elapsed time: 3.1 seconds
after doublet removal: 2,151 cells
after MT filter: 2,151 cells


In [ ]:
# STUDY 3 CAN BE DROPPED DUE TO INSUFFICIENT METADATA

In [47]:
# STUDY 4

In [50]:
study_name = 'Song2022'
paths = study_files[study_name]

# matrix,gene and cell
adata = sc.read_mtx(paths['matrix']).T
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes

cells = pd.read_csv(paths["cells"], index_col=0)
adata.obs_names = cells.index
adata.obs_names = adata.obs_names.astype(str) 
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"{adata.n_obs:,} cells, {adata.n_vars:,} genes")

21,743 cells, 21,877 genes


In [51]:
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease'] #check if present

# combine all cols that are in the study
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]

drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    
    # 1. NaN (for any dtype)
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask = drop_mask | nan_mask
    
    # 2. For object / string columns, "unknown", "NA", "nan", empty
    if vals.dtype == object:
        # Convert to string, strip and lowercase
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} values to be dropped")
            drop_mask = drop_mask | suspect

print(f"cells flagged: {drop_mask.sum()}")

# drop
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"cells remaining after drop: {adata.n_obs}")
else:
    print('no cells dropped')

cells flagged: 0
no cells dropped


In [52]:
# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
min_genes = 200
max_genes = 6000
keep = (adata.obs["n_genes_by_counts"] >= min_genes) & \
       (adata.obs["n_genes_by_counts"] <= max_genes)
adata = adata[keep, :].copy()
print(f"after gene filter {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"after doublet removal: {adata.n_obs:,} cells")

# Dead cell removal
adata.var['mt'] = adata.var_names.str.startswith('MT-') #get mitochondrial genes
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
max_pct_mt = 15
keep_mt = adata.obs['pct_counts_mt'] <= max_pct_mt
adata = adata[keep_mt, :].copy()
print(f"after MT filter: {adata.n_obs:,} cells")

# conversion to string
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

adata.write_h5ad('prostate_study4_qc.h5ad')

after gene filter 21,540 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.72
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 0.5%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 3.4%
Elapsed time: 20.6 seconds
after doublet removal: 21,536 cells
after MT filter: 20,355 cells


In [59]:
# NORMALIZE ALL STUDIES

In [60]:
import glob

In [61]:
files = {
    "prostate_study1_qc.h5ad": "Chen2021",
    "prostate_study2_qc.h5ad": "Dong2020",
    "prostate_study3_qc.h5ad": "He2021",
    "prostate_study4_qc.h5ad": "Song2022"
}

for fname, study_name in files.items():
    unit = study_files[study_name]["units"]
    adata = sc.read_h5ad(fname)

    if unit == "TPM":
        sc.pp.log1p(adata)
    else:
        sc.pp.normalize_total(adata, target_sum=1e6)
        sc.pp.log1p(adata)

    out_name = f"{study_name}_normalized.h5ad"
    adata.write_h5ad(out_name)
    print(f"{study_name}: normalised ({unit}) – max value {adata.X.max():.2f}")

Chen2021: normalised (UMI) – max value 13.16
Dong2020: normalised (UMI) – max value 13.54
He2021: normalised (TPM) – max value 12.98
Song2022: normalised (UMI) – max value 12.87
